In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import sys
sys.path.append("/storage/groups/ml01/workspace/jan.domke/Code/")
import scib_excerpts

Import dataset with clean metadata from Lisa, to obtain same columns for all datasets.

In [2]:
clean_metadata = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/query_dataset_scarches_input.h5ad")

In [3]:
clean_metadata.obs["condition"] = clean_metadata.obs["condition"].astype("string")

In [4]:
clean_metadata.obs.loc[clean_metadata.obs.disease=="IPF","condition"] = "Endstage lung fibrosis (IPF)"
clean_metadata.obs.loc[clean_metadata.obs.disease=="EAA","condition"] = "Endstage lung fibrosis (EAA)"

In [5]:
clean_metadata.obs["condition"] = clean_metadata.obs["condition"].astype("category")

In [6]:
#Import mapping from gene symbols to gene ids

In [7]:
mapping_gene_ids_symbols = pd.read_csv("/storage/groups/ml01/workspace/jan.domke/Data/Homo_sapiens_GRCh38_84_gene_ids_to_gene_symbol.csv")

In [8]:
mapping_gene_ids_symbols.rename(columns={"Unnamed: 0":"gene_id"},inplace=True)

In [9]:
mapping_gene_ids_symbols_short = mapping_gene_ids_symbols.sort_values("gene_id").groupby("gene_name", as_index=False).first()

In [10]:
#Rename double gene_name from mapping_gene_ids_symbols and append -1 to second one from each pair
double_index = (mapping_gene_ids_symbols.groupby("gene_name").count()>1)[(mapping_gene_ids_symbols.groupby("gene_name").count()>1)["gene_id"]].index
mask_rename = ~mapping_gene_ids_symbols.gene_id.isin(mapping_gene_ids_symbols_short.gene_id)
mapping_gene_ids_symbols.loc[mask_rename,"gene_name"] = mapping_gene_ids_symbols.loc[mask_rename,"gene_name"] + "-1"

In [11]:
#Import gene_ids to keep

In [12]:
gene_ids_to_keep = pd.read_csv("/storage/groups/ml01/workspace/jan.domke/Data/Homo_sapiens_GRCh38_84_gene_ids_to_keep_cellrangerbased.txt", header=None)

In [13]:
gene_ids_to_keep.rename(columns={0:"gene_id"},inplace=True)
gene_ids_to_keep = gene_ids_to_keep.gene_id.tolist()

In [14]:
#Import gene_names to keep

In [15]:
gene_names_to_keep = pd.read_csv("/storage/groups/ml01/workspace/jan.domke/Data/Homo_sapiens_GRCh38_84_gene_names_to_keep_cellrangerbased.txt", header=None)

In [16]:
gene_names_to_keep.rename(columns={0:"gene_name"},inplace=True)
gene_names_to_keep = gene_names_to_keep.gene_name.tolist()

## 0. Healthy data

In [17]:
healthy_data = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/LCA_h5ads/Meyer_2021_raw.h5ad")

In [18]:
healthy_data.obs.index = healthy_data.obs.index+"_meyer"

In [19]:
print("# genes to keep: ", np.sum(healthy_data.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(healthy_data.var.index))
print("ratio of genes to keep: ", np.sum(healthy_data.var.index.isin(gene_ids_to_keep))/len(healthy_data.var.index))

# genes to keep:  33694
# genes total:  33694
ratio of genes to keep:  1.0


In [20]:
healthy_data.obs = pd.concat([healthy_data.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(healthy_data.obs.columns):]

## 1. Nucleus data

In [21]:
query_data1 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/lukassen.h5ad")
query_data2 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/duong.h5ad")
query_data3 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch1.h5ad")
query_data4 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch2.h5ad")
query_data5 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch3.h5ad")
query_data6 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/wang_sub_batch4.h5ad")

In [22]:
query_data1.var["gene_name"] = query_data1.var.index
query_data1.var = query_data1.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data1.var.index = query_data1.var.gene_name

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [23]:
print("# genes to keep: ", np.sum(query_data1.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data1.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data1.var.index.isin(gene_names_to_keep))/len(query_data1.var.index),2))

# genes to keep:  30299
# genes total:  32738
ratio of genes to keep:  0.93


In [24]:
query_data2.var["gene_name"] = query_data2.var.index
query_data2.var = query_data2.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data2.var.index = query_data2.var.gene_name

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [25]:
print("# genes to keep: ", np.sum(query_data2.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data2.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data2.var.index.isin(gene_names_to_keep))/len(query_data2.var.index),2))

# genes to keep:  19192
# genes total:  27678
ratio of genes to keep:  0.69


In [26]:
query_data3.var["gene_name"] = query_data3.var.index
query_data3.var = query_data3.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data3.var.index = query_data3.var.gene_name

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [27]:
print("# genes to keep: ", np.sum(query_data3.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data3.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data3.var.index.isin(gene_names_to_keep))/len(query_data3.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [28]:
query_data4.var["gene_name"] = query_data4.var.index
query_data4.var = query_data4.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data4.var.index = query_data4.var.gene_name

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [29]:
print("# genes to keep: ", np.sum(query_data4.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data4.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data4.var.index.isin(gene_names_to_keep))/len(query_data4.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [30]:
query_data5.var["gene_name"] = query_data5.var.index
query_data5.var = query_data5.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data5.var.index = query_data5.var.gene_name

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [31]:
print("# genes to keep: ", np.sum(query_data5.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data5.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data5.var.index.isin(gene_names_to_keep))/len(query_data5.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [32]:
query_data6.var["gene_name"] = query_data6.var.index
query_data6.var = query_data6.var.merge(mapping_gene_ids_symbols, on="gene_name",how="left").copy()
query_data6.var.index = query_data6.var.gene_name

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [33]:
print("# genes to keep: ", np.sum(query_data6.var.index.isin(gene_names_to_keep)))
print("# genes total: ", len(query_data6.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data6.var.index.isin(gene_names_to_keep))/len(query_data6.var.index),2))

# genes to keep:  18809
# genes total:  26578
ratio of genes to keep:  0.71


In [34]:
query_data1.obs.index = query_data1.obs.index+"_lukassen"
query_data2.obs.index = query_data2.obs.index+"_duong"
query_data3.obs.index = query_data3.obs.index+"-sub_wang_sub_batch1"
query_data4.obs.index = query_data4.obs.index+"-sub_wang_sub_batch2"
query_data5.obs.index = query_data5.obs.index+"-sub_wang_sub_batch3"
query_data6.obs.index = query_data6.obs.index+"-sub_wang_sub_batch4"

#### 1.1 Get clean metadata for each dataset

In [35]:
query_data1.obs = pd.concat([query_data1.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data1.obs.columns):]
query_data2.obs = pd.concat([query_data2.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data2.obs.columns):]
query_data3.obs = pd.concat([query_data3.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data3.obs.columns):]
query_data4.obs = pd.concat([query_data4.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data4.obs.columns):]
query_data5.obs = pd.concat([query_data5.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data5.obs.columns):]
query_data6.obs = pd.concat([query_data6.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data6.obs.columns):]

In [36]:
query_data_tmp1 = query_data1.concatenate(query_data2, join="outer")
query_data_tmp2 = query_data_tmp1.concatenate(query_data3, join="outer")
query_data_tmp3 = query_data_tmp2.concatenate(query_data4, join="outer")
query_data_tmp4 = query_data_tmp3.concatenate(query_data5, join="outer")
nucleus_data = query_data_tmp4.concatenate(query_data6, join="outer")

#### 1.2 Get cell types for annotation harmonization

Sent to Lisa for cell type label harmonisation.

## 2. IPF data

In [17]:
query_data1 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/adams.h5ad")
query_data2 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/haberman.h5ad")
query_data3 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/reyfman_disease.h5ad")
query_data4 = sc.read("/storage/groups/ml01/workspace/hlca_lisa.sikkema_malte.luecken/query_datasets/ready/full/tsukui.h5ad")

In [18]:
query_data1.var.index=query_data1.var.ensembl

In [19]:
print("# genes to keep: ", np.sum(query_data1.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data1.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data1.var.index.isin(gene_ids_to_keep))/len(query_data1.var.index),2))

# genes to keep:  30924
# genes total:  45947
ratio of genes to keep:  0.67


In [40]:
# Rename gene_names in query_data2 from xxx.1 to xxx-1
query_data2.var["gene_name"] = query_data2.var.index
tmp = query_data2.var.merge(mapping_gene_ids_symbols, on="gene_name", how="left")
mask_rename_genes_query2 = query_data2.var.gene_name.isin(tmp[tmp.gene_id.isna()].gene_name)
query_data2.var.loc[mask_rename_genes_query2,"gene_name"] = query_data2.var[mask_rename_genes_query2].gene_name.str[:-2]+"-1"

In [41]:
query_data2.var = query_data2.var.merge(mapping_gene_ids_symbols, on="gene_name")
query_data2.var.index = query_data2.var.gene_id

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [42]:
print("# genes to keep: ", np.sum(query_data2.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data2.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data2.var.index.isin(gene_ids_to_keep))/len(query_data2.var.index),2))

# genes to keep:  33694
# genes total:  33694
ratio of genes to keep:  1.0


In [43]:
query_data3.var["gene_name"] = query_data3.var.index
query_data3.var = query_data3.var.merge(mapping_gene_ids_symbols, on="gene_name").copy()
query_data3.var.index = query_data3.var.gene_id

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [44]:
print("# genes to keep: ", np.sum(query_data3.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data3.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data3.var.index.isin(gene_ids_to_keep))/len(query_data3.var.index),2))

# genes to keep:  27181
# genes total:  27181
ratio of genes to keep:  1.0


In [45]:
query_data4.var["gene_name"] = query_data4.var.index
query_data4.var = query_data4.var.merge(mapping_gene_ids_symbols, on="gene_name")
query_data4.var.index = query_data4.var.gene_id

AnnData expects .var.index to contain strings, but your first indices are: Int64Index([0, 1], dtype='int64'), …


In [46]:
print("# genes to keep: ", np.sum(query_data4.var.index.isin(gene_ids_to_keep)))
print("# genes total: ", len(query_data4.var.index))
print("ratio of genes to keep: ", np.round(np.sum(query_data4.var.index.isin(gene_ids_to_keep))/len(query_data4.var.index),2))

# genes to keep:  27147
# genes total:  27147
ratio of genes to keep:  1.0


In [47]:
query_data1.obs.index = query_data1.obs.index+"_adams"
query_data2.obs.index = query_data2.obs.index+"_haberman"
query_data3.obs.index = query_data3.obs.index+"_reyfman_disease"
query_data4.obs.index = query_data4.obs.index+"_tsukui"

#### 2.1 Get clean metadata for each dataset

In [48]:
query_data1.obs = pd.concat([query_data1.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data1.obs.columns):]
query_data2.obs = pd.concat([query_data2.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data2.obs.columns):]
query_data3.obs = pd.concat([query_data3.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data3.obs.columns):]
query_data4.obs = pd.concat([query_data4.obs, clean_metadata.obs], axis=1,join="inner").iloc[:,len(query_data4.obs.columns):]

Rename .obs.dataset for Banovich_Kropski_2020 (haberman) and Misharin_Budinger_2018 (reyfman_disease) so that they get treated as different batch

In [49]:
query_data2.obs.dataset = query_data2.obs.dataset.cat.rename_categories({"Banovich_Kropski_2020":"Banovich_Kropski_2020_disease"})
query_data3.obs.dataset = query_data3.obs.dataset.cat.rename_categories({"Misharin_Budinger_2018":"Misharin_Budinger_2018_disease"})

In [50]:
query_data_tmp1 = query_data1.concatenate(query_data2, join="outer")
query_data_tmp2 = query_data_tmp1.concatenate(query_data3, join="outer")
ipf_data = query_data_tmp2.concatenate(query_data4, join="outer")

#### 1.2 Get cell types for annotation harmonization

Sent to Lisa for cell type label harmonisation.

## 3. Save them

In [51]:
ipf_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Data_to_harmonize/ipf_data_to_harmonize.h5ad")
nucleus_data.write("/storage/groups/ml01/workspace/jan.domke/Data/Data_to_harmonize/nucleus_data_to_harmonize.h5ad")

... storing 'dataset' as categorical
... storing 'ensembl-0-0-0' as categorical
... storing 'symbol-0-0-0' as categorical
... storing 'gene_name-1-0-0' as categorical
... storing 'gene_id-1-0-0' as categorical
... storing 'gene_name-1-0' as categorical
... storing 'gene_id-1-0' as categorical
... storing 'gene_name-1' as categorical
... storing 'gene_id-1' as categorical
... storing 'name-0-0-0-0-0' as categorical
... storing 'gene_name-0-0-0-0-0' as categorical
... storing 'gene_id-0-0-0-0-0' as categorical
... storing 'gene_name-1-0-0-0-0' as categorical
... storing 'gene_id-1-0-0-0-0' as categorical
... storing 'features-1-0-0-0-0' as categorical
... storing 'gene_name-1-0-0-0' as categorical
... storing 'gene_id-1-0-0-0' as categorical
... storing 'gene_name-1-0-0' as categorical
... storing 'gene_id-1-0-0' as categorical
... storing 'gene_name-1-0' as categorical
... storing 'gene_id-1-0' as categorical
... storing 'gene_name-1' as categorical
... storing 'gene_id-1' as categorica